[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FutoshiNakamura-Tts/gaussian-splatting-colab/blob/colab-t4-2025-12-18/gaussian_splatting_colab.ipynb)

In [ ]:
# @title 1. Key Imports & Utilities
import os
import sys
import shutil
import subprocess
import time
import re
import shlex
import torch
import glob
from google.colab import drive, files
from threading import Timer
from queue import Queue
from random import randint
import ipywidgets as widgets
from IPython.display import display, clear_output, Javascript

# Output Area for Logs
out = widgets.Output(layout={'border': '1px solid #ddd', 'height': '300px', 'overflow_y': 'scroll'})
out_viewer = widgets.Output(layout={'border': '1px solid #ddd', 'margin': '10px 0'})

def log(msg):
    # Use append_stdout for thread safety with ipywidgets.Output
    out.append_stdout(str(msg) + '\n')

def run_command(cmd, shell=True, env=None):
    try:
        process = subprocess.Popen(
            cmd, shell=shell, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            env=env
        )
        for line in process.stdout:
            log(line.strip())
        process.wait()
        if process.returncode != 0:
             log(f"Command failed with return code {process.returncode}")
    except Exception as e:
        log(f"Error executing command: {e}")


In [ ]:
# @title 2. Configuration
# ===========================
# Configuration Parameters
# ===========================
# @markdown ### Global Config
# (Updated: 2025-12-23 10:07:25)
Mount_Drive = False # @param {type:"boolean"}
Use_Tailscale = False # @param {type:"boolean"}

# @markdown ### Data Config
DataSource = "Demo Data" # @param ["Demo Data", "Google Drive", "Upload Zip", "Custom URL", "Local Folder"]
DrivePath = "/content/drive/MyDrive/my_data.zip" # @param {type:"string"}
Custom_URL = "" # @param {type:"string"}

Repo_Branch = "" # @param {type:"string"}

# @markdown ### Training Config
Source_Path = "/content/tandt/truck" # @param {type:"string"}
Output_Path = "/content/gaussian-splatting/output/my-experiment" # @param {type:"string"}
Iterations = 30000 # @param {type:"integer"}
SH_Degree = 3 # @param {type:"integer"}
White_Background = False # @param {type:"boolean"}
Eval_Mode = False # @param {type:"boolean"}
DryRun = False # @param {type:"boolean"}


In [ ]:
# @title 3. Define Actions
import shlex

def setup_env(b):
    out.clear_output()
    run_command(f"{sys.executable} scripts/setup_env.py")

def install_deps(b):
    out.clear_output()
    accel_flag = ""
    if 'dd_rasterizer' in globals() and dd_rasterizer.value.startswith('Accelerated'):
        accel_flag = "--accelerated"
    run_command(f"{sys.executable} scripts/install_deps.py {accel_flag}")


In [ ]:
def prepare_data(b):
    out.clear_output()
    source_type = dd_datasource.value if 'dd_datasource' in globals() else "Demo Data"
    path = txt_source_path.value if 'txt_source_path' in globals() else ""
    
    # Simple pass-through. prepare_data.py will handle validation.
    cmd = f"{sys.executable} scripts/prepare_data.py --source {shlex.quote(source_type)}"
    if path: cmd += f" --path {shlex.quote(path)}"
    run_command(cmd)


In [ ]:
def start_training(b):
    out.clear_output()
    # Gather Params
    src = Source_Path
    out_path = Output_Path
    # ... (Add logic to update Source_Path from GUI if needed, usually they modify globals or we read widgets)
    # But in Cell 4.1 'Main Execution', we export widgets to globals.
    # Let's read widgets directly if available.
    
    args = []
    args.append(f"--source_path {shlex.quote(src)}")
    args.append(f"--output_path {shlex.quote(out_path)}")
    args.append(f"--iterations {Iterations}")
    args.append(f"--sh_degree {SH_Degree}")
    if White_Background: args.append("--white_background")
    if Eval_Mode: args.append("--eval")
    
    # Widget Params
    if 'cb_antialiasing' in globals() and cb_antialiasing.value: args.append("--antialiasing")
    if 'cb_exposure' in globals() and cb_exposure.value: args.append("--exposure")
    if 'cb_depth' in globals() and cb_depth.value: 
        args.append("--depth")
        if 'txt_depth_path' in globals() and txt_depth_path.value:
             args.append(f"--depth_path {shlex.quote(txt_depth_path.value)}")
    if 'cb_sparse_adam' in globals() and cb_sparse_adam.value: args.append("--sparse_adam")
    if 'cb_dryrun' in globals() and cb_dryrun.value: args.append("--dry_run")
    
    # Transfer Params
    if 'drive' in sys.modules and os.path.exists('/content/drive'):
         # Maybe auto-backup to a folder with same name in Drive?
         # For now, just prompt or use default. Script handle it?
         # Let's check if the user set a destination in GUI? Currently no specific widget for dest.
         # We'll rely on script default or logic.
         pass
         
    # Tailscale Logic for Transfer
    # We need a widget for 'Tailscale Target Host'. 
    # We should add this widget in the GUI section (later refactor step) or just rely on hardcode/notebook var.
    # For now, let's assume valid notebook variable 'Tailscale_Target' if set.
    # Read dynamic value from widget if available
    ts_target = ""
    if 'txt_tailscale_target' in globals():
        ts_target = txt_tailscale_target.value
    elif 'Tailscale_Target' in globals():
        ts_target = Tailscale_Target
        
    if ts_target:
         args.append(f"--tailscale_target {shlex.quote(ts_target)}")
    
    full_cmd = f"{sys.executable} scripts/train.py " + " ".join(args)
    run_command(full_cmd)


In [ ]:
# @title 3.5. Tailscale Connection (Class)
# ===========================
# Tailscale Logic
# ===========================

class TailscaleConnection:
    def __init__(self):
        self.connected = False
        self.status_callback = None

    def log(self, msg):
        if self.status_callback:
             # Extract a simple status derived from the message if possible, or just pass 'Busy'
             # For now, we rely on specific status updates in methods
             pass
        # Use global log function from cell 1
        # Use global log function from cell 1
        if 'log' in globals():
            log(msg)
        else:
            print(msg)

    def install(self):
        if self.status_callback: self.status_callback("Installing...")
        if not os.path.exists('/usr/bin/tailscale'):
             self.log("Installing Tailscale...")
             run_command("curl -fsSL https://tailscale.com/install.sh | sh")

    def connect(self):
        if self.status_callback: self.status_callback("Connecting...")
        if self.connected:
            self.log("Tailscale already connected.")
            return
            
        self.log("\n=== Enabling Tailscale ===")
        self.install()
        
        self.log("Configuring Hostname: colab")
        run_command("hostname colab")
        
        self.log("Starting Tailscale Daemon...")
        run_command("nohup tailscaled --tun=userspace-networking --socket=/run/tailscale/tailscaled.sock --port 41641  >/dev/null 2>&1 &")
        
        self.log("Connecting...")
        run_command("tailscale up --ssh --hostname=colab")
        self.log("Tailscale Connected (check output for login link if needed).")
        self.log("[DEBUG] Verifying connection...")
        run_command("tailscale status")
        self.connected = True
        if self.status_callback: self.status_callback("Connected")

    def disconnect(self):
        if not self.connected:
            return
        self.log("\n=== Disabling Tailscale ===")
        run_command("tailscale down")
        self.log("Tailscale Disconnected.")

    def toggle(self, change):
        if change['new']:
            self.connect()
        else:
            self.disconnect()

tailscale_conn = TailscaleConnection()

# Auto-connect if configured globally (e.g. at start)
if 'Use_Tailscale' in globals() and Use_Tailscale:
    tailscale_conn.connect()


In [ ]:
# @title 4.1. GUI Layout (View)
# ===========================
# Defines widgets and layout
# ===========================
import ipywidgets as widgets
from IPython.display import display

class GUIWidgets:
    def __init__(self):
        self.style = {'description_width': 'initial'}
        self.layout = widgets.Layout(width='auto')
        
        # --- Status Indicator ---
        self.status_html = widgets.HTML(
            value='''
            <style>
            .status-box { padding: 5px; border-radius: 4px; font-weight: bold; }
            .status-ready { background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }
            .status-busy { background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }
            .loader { border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }
            @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
            <div class="status-box status-ready">✅ Ready</div>
            '''
        )

        # --- Core Widgets ---
        self.lbl_tailscale_status = widgets.Label(value="Disconnected", style={'text_color': 'gray'})
        self.cb_tailscale = widgets.Checkbox(value=False, description='Connect Tailscale (ssh)', style=self.style)
        self.txt_tailscale_target = widgets.Text(value='', placeholder='Hostname (e.g. mypc) for File Drop', description='Target Device:', style=self.style)
        self.cb_dryrun = widgets.Checkbox(value=False, description='Dry Run (No GPU)', style=self.style)
        
        self.btn_env = widgets.Button(description="1. Setup Environment", button_style='primary', layout=self.layout)
        self.dd_rasterizer = widgets.Dropdown(options=['Standard', 'Accelerated (Sparse Adam)'], value='Standard', description='Rasterizer:', style=self.style)
        self.btn_deps = widgets.Button(description="2. Install Dependencies", button_style='info', layout=self.layout)
        
        self.dd_datasource = widgets.Dropdown(options=['Demo Data', 'User Data (Path)'], value='Demo Data', description='Data Source:', style=self.style)
        self.txt_source_path = widgets.Text(value='', placeholder='Drive Path / URL / Folder Path', description='Path / URL:', style=self.style)
        self.btn_browse = widgets.Button(description="📂 Browse", layout=widgets.Layout(width='100px'))
        
        # File Browser Widgets (Source Path)
        self.lbl_browser_title = widgets.Label(value="Select File", style={'font_weight': 'bold'})
        self.lbl_path = widgets.Label(f"Current: /")
        self.sel_files = widgets.Select(options=[], rows=10, layout=widgets.Layout(width='100%'))
        self.btn_up = widgets.Button(description="⬆ Up", layout=widgets.Layout(width='80px'))
        self.btn_select = widgets.Button(description="Select", button_style='primary', layout=widgets.Layout(width='80px'))
        self.btn_cancel_browser = widgets.Button(description="Cancel", layout=widgets.Layout(width='80px'))
        
        self.browser_box = widgets.VBox([
            self.lbl_browser_title,
            widgets.HBox([self.btn_up, self.lbl_path]),
            self.sel_files,
            widgets.HBox([self.btn_cancel_browser, self.btn_select])
        ])
        self.browser_box.layout.display = 'none'


        self.btn_data = widgets.Button(description="3. Prepare Data", button_style='warning', layout=self.layout)
        
        # Training Options
        self.cb_antialiasing = widgets.Checkbox(value=False, description='Anti-aliasing', style=self.style)
        self.cb_exposure = widgets.Checkbox(value=False, description='Exposure Comp', style=self.style)
        self.cb_depth = widgets.Checkbox(value=False, description='Depth Reg', style=self.style)
        self.txt_depth_path = widgets.Text(value='', placeholder='Depth maps path', description='Depth Path:', display='none', style=self.style)
        self.txt_depth_path.layout.display = 'none'
        self.cb_sparse_adam = widgets.Checkbox(value=False, description='Sparse Adam', disabled=True, style=self.style)
        
        self.btn_train = widgets.Button(description="4. Train", button_style='success', layout=self.layout)
        
        # Viewer Widgets
        

        # --- Layout Container ---
        self.container = widgets.VBox([
            widgets.HBox([widgets.HTML("<h3>Gaussian Splatting Controller</h3>"), self.status_html]),
            widgets.HBox([self.cb_tailscale, self.lbl_tailscale_status, self.txt_tailscale_target]),
            widgets.HBox([self.btn_env]),
            widgets.HBox([self.dd_rasterizer, self.btn_deps]),
            widgets.HTML("<hr>"),
            widgets.HBox([self.dd_datasource, self.txt_source_path, self.btn_browse]),
            self.browser_box,
            widgets.HBox([self.btn_data]),
            widgets.HTML("<hr>"),
            widgets.Label("Training Options (New Features):"),
            widgets.HBox([self.cb_antialiasing, self.cb_exposure, self.cb_sparse_adam]),
            widgets.HBox([self.cb_depth, self.txt_depth_path]),
            widgets.HTML("<br>"),
            widgets.HBox([self.cb_dryrun]),
            
            widgets.HTML("<hr>"),
            widgets.HBox([self.btn_train]),
            widgets.HTML("<br>"),
        
            widgets.Label("Logs:"),
            out  # Using global 'out' widget
        ])

    def set_status(self, state, msg):
        if state == 'busy':
            self.status_html.value = f'''
            <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-busy {{ background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }} .loader {{ border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }} @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}</style>
            <div class="status-box status-busy"><div class="loader"></div> {msg}</div>'''
        else:
            self.status_html.value = '''
             <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-ready {{ background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }}</style>
            <div class="status-box status-ready">✅ Ready</div>'''

        # Lockable widgets during async actions
        # Exclude Tailscale from this list to keep it independent
        self.lockable_widgets = [
            self.cb_dryrun, self.btn_env, self.dd_rasterizer,
            self.btn_deps, self.dd_datasource, self.txt_source_path, self.btn_browse,
            self.btn_data, self.cb_antialiasing, self.cb_exposure,
        ]


In [ ]:
# @title 4.2. GUI Controller (Logic)
# ===========================
# Event binding and Logic
# ===========================
import os

class GUIController:
    def __init__(self, view):
        self.view = view
        self.current_path = os.getcwd()
        self.active_input = None
        
        # Init browsers
        self._update_browser(self.view.sel_files, self.view.lbl_path, self.current_path)

        # Bind events last to ensure all methods are defined
        self._bind_events()

    def _bind_events(self):
        # Tailscale
        if 'tailscale_conn' in globals():
            self.view.cb_tailscale.observe(self._on_tailscale_change, names='value')
            tailscale_conn.status_callback = lambda msg: setattr(self.view.lbl_tailscale_status, 'value', msg)
        
        # Visibility Logic
        self.view.dd_rasterizer.observe(self._on_rasterizer_change, names='value')
        self.view.cb_depth.observe(self._on_depth_change, names='value')
        self.view.dd_datasource.observe(self._on_datasource_change, names='value')
        
        # File Browser (Source)
        self.view.btn_browse.on_click(lambda b: self._open_browser_source(self.view.txt_source_path, "Source Path"))
        self.view.btn_up.on_click(lambda b: self._on_up(self.view.sel_files, self.view.lbl_path))
        self.view.sel_files.observe(lambda change: self._on_select_item(change, self.view.sel_files, self.view.lbl_path), names='value')
        self.view.btn_select.on_click(lambda b: self._on_confirm_select(self.view.browser_box, self.view.sel_files))
        self.view.btn_cancel_browser.on_click(lambda b: self._on_cancel_browser(self.view.browser_box))

        # Actions with Async Feedback
        self.view.btn_env.on_click(lambda b: self._run_action("Setting up Env...", setup_env))
        self.view.btn_deps.on_click(lambda b: self._run_action("Installing Deps...", install_deps))
        self.view.btn_data.on_click(lambda b: self._run_action("Preparing Data...", prepare_data))
        self.view.btn_train.on_click(lambda b: self._run_action("Training...", start_training))

    def _run_action(self, msg, func):
        self.view.set_status('busy', msg)
        for w in self.view.lockable_widgets:
            w.disabled = True

        try:
            func(None)
        except Exception as e:
            log(f"Error: {e}")
        finally:
            self.view.set_status('ready', "")
            for w in self.view.lockable_widgets:
                w.disabled = False
    
    def _on_tailscale_change(self, change):
        if 'tailscale_conn' in globals():
            tailscale_conn.toggle(change)

    # --- Event Handlers ---
    def _on_rasterizer_change(self, change):
        if change['new'].startswith('Accelerated'):
            self.view.cb_sparse_adam.value = True
        else:
            self.view.cb_sparse_adam.value = False

    def _on_depth_change(self, change):
        if change['new']:
            self.view.txt_depth_path.layout.display = 'flex'
        else:
            self.view.txt_depth_path.layout.display = 'none'

    def _on_datasource_change(self, change):
        val = change['new']
        if val == 'User Data (Path)':
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'block'
        else: # Demo Data
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
    def _on_datasource_change(self, change):
        val = change['new']
        if val in ['Google Drive', 'Local Folder']:
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'block'
        elif val == 'Upload Zip':
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
        elif val == 'Custom URL':
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'none'
        else: # Demo Data
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'

    # --- Browser Logic ---
    def _update_browser(self, widget_sel, widget_lbl, path):
        try:
            if not os.path.exists(path): path = '/content'
            items = sorted(os.listdir(path))
            formatted_items = []
            for item in items:
                if os.path.isdir(os.path.join(path, item)):
                    formatted_items.append(f"\ud83d\udcc2 {item}")
                else:
                    formatted_items.append(f"\ud83d\udcc4 {item}")
            widget_sel.options = formatted_items
            widget_lbl.value = f"Current: {path}"
        except Exception as e:
            widget_lbl.value = f"Error: {e}"

    def _on_up(self, widget_sel, widget_lbl):
        self.current_path = os.path.dirname(self.current_path)
        self._update_browser(widget_sel, widget_lbl, self.current_path)

    def _on_select_item(self, change, widget_sel, widget_lbl):
        if change['new']:
            name = change['new'].split(' ', 1)[1]
            full_path = os.path.join(self.current_path, name)
            if os.path.isdir(full_path):
                 self.current_path = full_path
                 self._update_browser(widget_sel, widget_lbl, self.current_path)

    def _on_confirm_select(self, browse_box_widget, widget_sel):
        val = widget_sel.value
        path_to_use = self.current_path
        
        if val:
            name = val.split(' ', 1)[1]
            path_to_use = os.path.join(path_to_use, name)
        
        if self.active_input:
            self.active_input.value = path_to_use
        
        browse_box_widget.layout.display = 'none'
        self.active_input = None

    def _open_browser_source(self, target_widget, title="Select File"):
        self.active_input = target_widget
        self.view.lbl_browser_title.value = f"Selecting: {title}"
        self.view.browser_box.layout.display = 'block'
        self._update_browser(self.view.sel_files, self.view.lbl_path, self.current_path)

    def _on_cancel_browser(self, browse_box_widget):
        browse_box_widget.layout.display = 'none'
        self.active_input = None


In [ ]:
# @title 5. Main Execution
if __name__ == "__main__":
    # 1. Instantiate View
    view = GUIWidgets()
    
    # 2. Restore logic from Cell 4.2 (Global Exports & Status)
    if 'tailscale_conn' in globals() and tailscale_conn.connected:
        view.cb_tailscale.value = True

    # Export widgets to globals for action functions
    globals()['cb_tailscale'] = view.cb_tailscale
    globals()['Tailscale_Target'] = view.txt_tailscale_target.value # Initial value usually empty, but we bind it?
    # Note: text widgets need to be accessed via value at runtime usually, but our refactor code reads globals().
    # To make 'Tailscale_Target' dynamic, we should likely read view.txt_tailscale_target.value INSIDE the action function.
    # But our refactor_nb.py assumed globals(). Let's fix main execution to NOT export static value but maybe we can just alias the widget if we want.
    # Actually, the action function `start_training` in refactor_nb.py checks `if 'Tailscale_Target' in globals()`.
    # If we want it to be dynamic, we should alias the WIDGET to a global name, then read .value.
    globals()['txt_tailscale_target'] = view.txt_tailscale_target
    globals()['cb_dryrun'] = view.cb_dryrun
    globals()['dd_datasource'] = view.dd_datasource
    globals()['txt_source_path'] = view.txt_source_path
    globals()['dd_rasterizer'] = view.dd_rasterizer
    globals()['cb_antialiasing'] = view.cb_antialiasing
    globals()['cb_exposure'] = view.cb_exposure
    globals()['cb_depth'] = view.cb_depth
    globals()['txt_depth_path'] = view.txt_depth_path
    globals()['cb_sparse_adam'] = view.cb_sparse_adam
    
    # 3. Instantiate Controller
    controller = GUIController(view)
    
    # 4. Initialize State
    controller._on_datasource_change({'new': view.dd_datasource.value})
    
    # 5. Display GUI
    display(view.container)
